# Week 7 — Regression for Engineering Prediction

**ESC 113: Computer Aided Analysis for Engineering**

This notebook is designed to function as both a **lecture guide and a self-study resource**. Explanations are intentionally detailed so that common student questions are answered directly beside the code.

### Main engineering question

> Can we predict the net electrical power output of a combined-cycle power plant from measured operating and environmental conditions?

We will answer this using **multiple linear regression**.

## Table of Contents

1. What Is Regression?
2. Why Engineers Use Regression
3. The Combined Cycle Power Plant Dataset
4. What Does One Row Represent?
5. Feature Definitions and Units
6. Loading the Dataset
7. Renaming Features Clearly
8. Inspecting the Dataset
9. Features X and Target y
10. Exploratory Relationships
11. Train/Test Split
12. Why We Do Not Train on All Data
13. Build a Linear Regression Model
14. What model.fit() Actually Does
15. Make Predictions
16. MAE
17. RMSE
18. R²
19. Why We Need More Than One Metric
20. Actual vs Predicted Plot
21. Residuals
22. Largest Prediction Errors
23. Model Coefficients
24. Why Coefficients Are Not Automatically Feature Importance
25. Single-Feature vs Multiple-Feature Regression
26. Common Student Questions
27. Common Mistakes
28. Guided Practice
29. Independent Engineering Exercises
30. Review Quiz

## 1. What Is Regression?

**Regression** is used when the quantity we want to predict is numerical and continuous.

Examples:

- electrical power output in MW;
- concrete compressive strength in MPa;
- battery capacity in Ah;
- bridge deflection in mm;
- temperature in °C;
- energy consumption in kWh.

In this lesson, our target is **net electrical power output in MW**, so this is a regression problem.

### Student question: Why is this not classification?

Classification predicts categories such as:

- Fault / No Fault
- Safe / Unsafe
- Low / Medium / High

Regression predicts a numerical value such as **452.6 MW**.

## 2. Why Engineers Use Regression

Regression helps engineers:

- predict system performance;
- estimate quantities that are expensive or difficult to measure directly;
- understand relationships among operating variables;
- create fast surrogate models;
- support design and operational decisions.

However, a regression model is a **statistical model**, not a replacement for physical laws or engineering judgment.

## 3. Dataset — Combined Cycle Power Plant

**Source:** UCI Machine Learning Repository  
**Dataset ID:** 294  
**Dataset:** Combined Cycle Power Plant  
**Observations:** 9,568  
**Input features:** 4  
**Target:** Net electrical power output  
**Missing values:** None reported by UCI  
**Collection period:** 2006–2011  
**Operating condition:** Full-load operation

A combined-cycle power plant uses both **gas turbines** and **steam turbines**. Waste heat from the gas-turbine stage is recovered to generate steam, improving overall efficiency.

The dataset contains hourly average measurements collected from the plant while it was operating at full load.

**Reference:** UCI Combined Cycle Power Plant dataset  
https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant  
DOI: 10.24432/C5002N

## 4. What Does One Row Represent?

Each row represents **one hourly operating observation** of the power plant.

For that observation, we know:

- ambient temperature;
- exhaust vacuum;
- ambient pressure;
- relative humidity;
- measured net electrical output.

### Student question: Is each row a different power plant?

No. The rows are repeated operating observations from a combined-cycle power plant across time.

### Student question: Are these instantaneous measurements?

The UCI description states that the variables are **hourly averages** derived from sensor measurements.

## 5. Feature Definitions and Units

| Original UCI name | Descriptive name used here | Meaning | Unit | Role |
|---|---|---|---|---|
| AT | Ambient_Temperature_C | Outside air temperature | °C | Feature |
| V | Exhaust_Vacuum_cmHg | Steam-turbine exhaust vacuum | cm Hg | Feature |
| AP | Ambient_Pressure_mbar | Atmospheric pressure | mbar | Feature |
| RH | Relative_Humidity_percent | Relative humidity | % | Feature |
| PE | Net_Electrical_Output_MW | Net electrical power output | MW | Target |

### Why rename the columns?

Short names such as `AT` and `PE` are convenient for a dataset creator but are easy to forget during analysis.

Self-descriptive names reduce mistakes and make code easier to understand.

## 6. Install and Import Packages

In [ ]:
!pip -q install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 7. Load the Dataset

In [ ]:
# UCI dataset ID 294
power_plant = fetch_ucirepo(id=294)

X_raw = power_plant.data.features.copy()
y_raw = power_plant.data.targets.copy()

print("Raw feature shape:", X_raw.shape)
print("Raw target shape :", y_raw.shape)

display(X_raw.head())
display(y_raw.head())

### Student question: What does shape mean?

For a DataFrame:

`(rows, columns)`

So:

`(9568, 4)`

means **9,568 observations and 4 input features**.

## 8. Rename Features to Self-Descriptive Names

In [ ]:
X = X_raw.copy()
y = y_raw.copy()

X.columns = [
    "Ambient_Temperature_C",
    "Exhaust_Vacuum_cmHg",
    "Ambient_Pressure_mbar",
    "Relative_Humidity_percent"
]

y.columns = ["Net_Electrical_Output_MW"]

display(X.head())
display(y.head())

## 9. Inspect the Dataset Before Modeling

Never jump directly from loading data to machine learning.

First ask:

- Are the dimensions correct?
- Are there missing values?
- Are the data types reasonable?
- Are values physically plausible?
- Is the target numerical?
- Are there duplicated observations?

In [ ]:
print("Features:", X.shape)
print("Target  :", y.shape)

print("\nData types:")
print(X.dtypes)

print("\nMissing feature values:")
print(X.isna().sum())

print("\nMissing target values:")
print(y.isna().sum())

print("\nDuplicate feature rows:", X.duplicated().sum())

## 10. Descriptive Statistics

In [ ]:
display(X.describe())
display(y.describe())

### How to read describe()

For each numerical variable:

- **count** = number of non-missing observations
- **mean** = arithmetic average
- **std** = standard deviation
- **min** = minimum
- **25%** = first quartile
- **50%** = median
- **75%** = third quartile
- **max** = maximum

### Student question: Why inspect min and max?

They help identify impossible values, unit problems, or potential outliers before modeling.

## 11. Define X and y

Machine-learning convention:

- **X** = input features
- **y** = target we want to predict

Here:

X = environmental/operating measurements  
y = net electrical output in MW

In [ ]:
target = y["Net_Electrical_Output_MW"]

print("X shape:", X.shape)
print("y shape:", target.shape)

### Student question: Why is X uppercase but y lowercase?

It is a common convention:

- `X` usually represents a matrix with multiple feature columns.
- `y` usually represents one target vector.

Python does not require these names. They are simply widely used.

## 12. Explore Feature–Target Relationships

Before fitting a linear model, visually inspect whether relationships appear roughly linear.

In [ ]:
feature_labels = {
    "Ambient_Temperature_C": "Ambient Temperature (°C)",
    "Exhaust_Vacuum_cmHg": "Exhaust Vacuum (cm Hg)",
    "Ambient_Pressure_mbar": "Ambient Pressure (mbar)",
    "Relative_Humidity_percent": "Relative Humidity (%)"
}

for feature, label in feature_labels.items():
    plt.figure(figsize=(6,4))
    plt.scatter(X[feature], target, alpha=0.35)
    plt.xlabel(label)
    plt.ylabel("Net Electrical Output (MW)")
    plt.title(f"{label} vs Net Electrical Output")
    plt.grid(True)
    plt.show()

### Questions to ask while looking at these plots

1. Does output generally increase or decrease with the feature?
2. Does the relationship look approximately linear?
3. Is there substantial scatter?
4. Are there unusual points?
5. Could multiple features be interacting?

### Important

A visible association does **not prove causation**.

## 13. Train/Test Split

We will use:

- 80% for training
- 20% for testing
- `random_state=42` for reproducibility

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    target,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 14. Why Do We Need a Test Set?

The model learns from the training set.

The test set acts like an **unseen exam**.

If we evaluate only on the same data used for training, we cannot properly judge how well the model generalizes.

### Student question: Why not train using all 9,568 rows?

Because then we would have no independent observations left to evaluate performance.

### Student question: Why exactly 80/20?

There is no universal law requiring 80/20. It is a common practical choice.

Other splits such as 70/30 or 75/25 can also be reasonable depending on dataset size and study design.

### Student question: Does 42 have special mathematical meaning?

No. It is simply a fixed random seed. Any fixed integer can be used. The purpose is reproducibility.

## 15. Create the Linear Regression Model

In [ ]:
model = LinearRegression()

print(model)

At this point, the model object exists, but it has **not learned anything yet**.

It does not know the coefficients until we call:

`model.fit(...)`

## 16. Train the Model

In [ ]:
model.fit(X_train, y_train)

### What does model.fit(X_train, y_train) do?

The model searches for regression coefficients that best connect the input features to the known training targets.

For multiple linear regression:

**Predicted Power = intercept + b₁x₁ + b₂x₂ + b₃x₃ + b₄x₄**

The coefficients are chosen to reduce prediction error on the training data.

### Important

`X_test` and `y_test` are **not** used to fit the model.

## 17. Make Predictions on Unseen Test Data

In [ ]:
y_pred = model.predict(X_test)

print("First five predictions:")
print(y_pred[:5])

print("\nFirst five actual values:")
print(y_test.iloc[:5].to_numpy())

### Student question: What is model.predict() doing?

For each row in `X_test`, the model inserts the feature values into the learned regression equation and calculates a predicted electrical output.

### Student question: Does predict() change the model?

No. Prediction uses the already-fitted model.

## 18. Put Actual and Predicted Values Side by Side

In [ ]:
results = pd.DataFrame({
    "Actual_MW": y_test.to_numpy(),
    "Predicted_MW": y_pred
})

results["Residual_MW"] = results["Actual_MW"] - results["Predicted_MW"]
results["Absolute_Error_MW"] = np.abs(results["Residual_MW"])

display(results.head(10))

## 19. Mean Absolute Error — MAE

MAE is:

**average(|actual − predicted|)**

It answers:

> On average, how many MW away are our predictions from the measured output?

Because the target is measured in MW, **MAE is also measured in MW**.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE = {mae:.3f} MW")

### Engineering interpretation

If MAE were 3.5 MW, we would say:

> The model's predictions differ from measured plant output by about 3.5 MW on average.

That does **not** automatically tell us whether the error is acceptable. Acceptability depends on the engineering application.

## 20. Root Mean Squared Error — RMSE

RMSE also measures prediction error in MW, but it penalizes larger errors more strongly because errors are squared before averaging.

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE = {rmse:.3f} MW")

### Student question: Why is RMSE usually larger than MAE?

Large individual errors receive more weight in RMSE.

Therefore, RMSE is especially useful when large misses are undesirable.

## 21. Coefficient of Determination — R²

R² measures how much of the variation in the target is explained by the model.

Typical interpretation:

- R² = 1 → perfect fit
- R² close to 1 → strong predictive fit
- R² ≈ 0 → little improvement over predicting the mean
- R² < 0 → worse than a simple mean prediction on that evaluation set

In [ ]:
r2 = r2_score(y_test, y_pred)
print(f"R² = {r2:.3f}")

### Student question: Does R² = 0.90 mean the model is 90% accurate?

No.

R² is **not percent accuracy**.

It describes explained variation, not the fraction of predictions that are correct.

## 22. Why Do We Need MAE, RMSE, and R²?

They answer different questions:

| Metric | Main question | Unit |
|---|---|---|
| MAE | What is the typical absolute prediction error? | MW |
| RMSE | How strongly are larger errors affecting performance? | MW |
| R² | How much target variation is explained? | Unitless |

Never judge an engineering model from only one metric.

In [ ]:
print(f"MAE  = {mae:.3f} MW")
print(f"RMSE = {rmse:.3f} MW")
print(f"R²   = {r2:.3f}")

## 23. Actual vs Predicted Plot

A perfect model would place every point exactly on the 45° line:

**Predicted = Actual**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.5)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high], "--")
plt.xlabel("Actual Net Electrical Output (MW)")
plt.ylabel("Predicted Net Electrical Output (MW)")
plt.title("Actual vs Predicted Power Output")
plt.grid(True)
plt.show()

### How to interpret this plot

- Points near the diagonal → good predictions
- Points far from the diagonal → larger errors
- Systematic curvature → possible nonlinearity
- Different error sizes across the range → possible non-constant variance

Do not judge the model only from how attractive the plot looks. Use numerical metrics too.

## 24. Residuals

We define:

**Residual = Actual − Predicted**

Therefore:

- positive residual → model underpredicted
- negative residual → model overpredicted
- residual near zero → prediction was close

In [ ]:
residuals = y_test.to_numpy() - y_pred

plt.figure(figsize=(7,4))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Net Electrical Output (MW)")
plt.ylabel("Residual = Actual − Predicted (MW)")
plt.title("Residual Plot")
plt.grid(True)
plt.show()

### Student question: Are negative residuals bad?

No.

Negative only means the prediction was above the actual measurement.

We are concerned about:

- very large residuals;
- systematic patterns;
- trends;
- changing spread;
- groups of observations the model handles poorly.

## 25. Find the Largest Prediction Errors

In [ ]:
largest_errors = results.sort_values(
    "Absolute_Error_MW",
    ascending=False
).head(10)

display(largest_errors)

### Why inspect individual errors?

A good average metric can hide a few very poor predictions.

Engineering applications often care about **worst-case behavior**, not only average behavior.

## 26. Model Intercept and Coefficients

In [ ]:
print(f"Intercept: {model.intercept_:.4f}")

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

display(coefficients)

## 27. How to Interpret a Regression Coefficient

For a linear model, a coefficient represents the model's estimated change in predicted output for a one-unit increase in that feature, **while the other model features are held constant**.

Example:

If the coefficient for Ambient Temperature were −2.0, the model would associate a 1°C increase in ambient temperature with approximately a 2 MW decrease in predicted output, holding the other included features constant.

### Important cautions

A coefficient does not automatically prove:

- causation;
- physical mechanism;
- feature importance.

Features have different units and may be correlated.

## 28. Why Can't We Compare Raw Coefficient Magnitudes Directly?

Suppose:

- temperature is measured in °C;
- pressure is measured around 1000 mbar;
- humidity is measured in percent.

Their numerical scales are different.

A coefficient of 5 for one variable is not automatically more important than a coefficient of 0.5 for another.

Later techniques such as standardization, feature importance, and permutation methods can help with comparison.

## 29. Single-Feature Regression: Ambient Temperature Only

This lets us ask:

> How well can temperature alone predict plant output?

In [ ]:
X_temp = X[["Ambient_Temperature_C"]]

X_temp_train, X_temp_test, y_temp_train, y_temp_test = train_test_split(
    X_temp,
    target,
    test_size=0.20,
    random_state=42
)

temp_model = LinearRegression()
temp_model.fit(X_temp_train, y_temp_train)

temp_pred = temp_model.predict(X_temp_test)

temp_mae = mean_absolute_error(y_temp_test, temp_pred)
temp_rmse = np.sqrt(mean_squared_error(y_temp_test, temp_pred))
temp_r2 = r2_score(y_temp_test, temp_pred)

print(f"Temperature-only MAE  = {temp_mae:.3f} MW")
print(f"Temperature-only RMSE = {temp_rmse:.3f} MW")
print(f"Temperature-only R²   = {temp_r2:.3f}")

## 30. Compare One Feature vs All Four Features

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Ambient Temperature Only", "All Four Features"],
    "MAE_MW": [temp_mae, mae],
    "RMSE_MW": [temp_rmse, rmse],
    "R2": [temp_r2, r2]
})

display(comparison)

### Engineering question

Does using all four measurements substantially improve prediction compared with temperature alone?

This is more informative than simply assuming that "more features must always be better."

## 31. Common Student Questions

### Why does linear regression have the word linear?

Because the prediction is a linear combination of the model coefficients and input variables.

### Does linear regression require every individual scatter plot to look perfectly straight?

No. But strong nonlinear structure can make a simple linear model inadequate.

### Can R² be negative?

Yes. On test data, a model can perform worse than predicting the test-set mean.

### Is a high R² enough to deploy the model?

No. We must also consider errors, operating range, external validation, physical plausibility, and intended use.

### Why don't we scale the variables here?

Ordinary linear regression can be fit without scaling. Scaling becomes especially important for some other algorithms and for comparing standardized coefficient effects.

### Does the model know physics?

No. It learns statistical relationships in the supplied data.

### Can we predict outside the observed data range?

Technically the model can return a number, but extrapolation may be unreliable because the model has not been validated there.

## 32. Common Mistakes

### Mistake 1 — Putting the target inside X

This creates data leakage.

### Mistake 2 — Training on the test set

The test set should remain unseen during fitting.

### Mistake 3 — Saying R² = 0.95 means 95% accuracy

That interpretation is incorrect.

### Mistake 4 — Ignoring units

MAE = 4 has little meaning without knowing that the unit is MW.

### Mistake 5 — Treating correlation or coefficients as causation

Statistical association is not sufficient evidence of causality.

### Mistake 6 — Looking only at average error

Inspect residuals and the largest errors too.

### Mistake 7 — Assuming a model is valid everywhere

The training data represent a particular operating regime: full-load observations from this power plant.

## 33. Guided Practice

Complete these with your instructor.

1. Calculate the mean and standard deviation of `Net_Electrical_Output_MW`.
2. Plot Ambient Pressure vs power output.
3. Fit a model using only Relative Humidity.
4. Calculate MAE, RMSE, and R².
5. Compare that model with the temperature-only model.
6. Find the test observation with the largest absolute residual.
7. Decide whether the largest error is practically important and explain what additional engineering information would be needed.

## 34. Independent Engineering Exercises

1. Fit a model using only `Exhaust_Vacuum_cmHg`.
2. Compare its R² with the temperature-only model.
3. Fit a model using Temperature + Vacuum.
4. Compare two-feature and four-feature models.
5. Change `random_state` from 42 to 1. Do the metrics change?
6. Change the test fraction from 20% to 30%.
7. Plot residuals for the temperature-only model.
8. Identify whether errors look random or structured.
9. Create a table with Actual, Predicted, Residual, and Absolute Error.
10. Write a five-sentence engineering interpretation of the final four-feature model.

## 35. Short Review Quiz

1. What type of target requires regression?
2. What does one row in this dataset represent?
3. What is the target variable and its unit?
4. Why do we separate training and test data?
5. What does `model.fit()` do?
6. What does `model.predict()` do?
7. What does MAE measure?
8. Why is RMSE more sensitive to large errors?
9. What does R² describe?
10. Why is R² not the same as percent accuracy?
11. Define a residual using the convention in this notebook.
12. What does a negative residual mean?
13. Why can raw coefficient magnitudes be misleading?
14. What is data leakage?
15. Why should an engineer avoid extrapolating far beyond the training-data range?

## 36. Key Takeaways

By the end of Week 7, you should understand the complete regression workflow:

**Engineering question → dataset → feature definitions → data inspection → X/y → train/test split → model → fit → predict → metrics → residuals → interpretation**

The most important idea is not simply how to run `LinearRegression()`.

It is how to decide whether a regression result is **understandable, reproducible, physically reasonable, and useful for an engineering problem**.